In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import kagglehub
from datasets import load_dataset
from sklearn.preprocessing import LabelEncoder
from tensorflow import keras
import tensorflow as tf
import tensorflow_addons as tfa
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense, Dropout, Input, Concatenate
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import classification_report, f1_score
import re
import unicodedata
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from imblearn.over_sampling import RandomOverSampler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_excel("../data/nnnhis_year_and_last_year_cleanedmix_shuffled1.xlsx")
df = df.rename(columns={"emotion_core":"emotion"})
df = df[["sentence", "emotion"]]
df["sentence"] = df["sentence"].astype(str)
df.head()

# Iteration 1: Baseline RNN

In [ ]:
# Encode emotions
label_encoder = LabelEncoder()
df["label"] = label_encoder.fit_transform(df["emotion"])

X = df["sentence"].values
y = df["label"].values

# split into train validation split
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size = 0.1, random_state=42, stratify=y
)

In [ ]:
max_words = 20000 # keep the 20000 most common words in the corpus so the model doesnt waste time training sentences that only happen onces
max_len = 100 # Fix input length for padding the vectors

# Build vocabulary from trainig data
tokenzier = Tokenizer(num_words=max_words, oov_token="<OOV>") # replace words out of the corpus with OOV
tokenzier.fit_on_texts(X_train)

# Convert sentences into integer sequences 
X_train_seq = tokenzier.texts_to_sequences(X_train)
X_val_seq = tokenzier.texts_to_sequences(X_val)

# pad to fix length issues
X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding="post", truncating="post")
X_val_pad = pad_sequences(X_val_seq, maxlen=max_len, padding="post", truncating="post")

In [ ]:
embedding_dim = 128

model_1 = Sequential([
    Embedding(input_dim = max_words, output_dim = embedding_dim, input_length = max_len),
    SimpleRNN(128, return_sequences=False),
    Dropout(0.5),
    Dense(64, activation = "relu"),
    Dropout(0.3),
    Dense(len(label_encoder.classes_), activation = "softmax")
])

model_1.compile(
    loss = "sparse_categorical_crossentropy",
    optimizer = "adam",
    metrics = ["accuracy"]
)

model_1.summary()

In [ ]:
h1 = model_1.fit(
    X_train_pad, y_train,
    validation_data = (X_val_pad, y_val),
    epochs = 5,
    batch_size = 32,
    verbose = 1
)

In [ ]:
# Loading and pre process test dataset
df_test = pd.read_csv("../data/group_23_url_1_transcript.csv")
# Keep only the necessary columns
df_test = df_test[["Sentence", "Emotion_core"]]
df_test.head()

In [ ]:
# Encode labels with the same encoder used in training
df_test["label"] = label_encoder.transform(df_test["Emotion_core"])

X_test = df_test["Sentence"].values
y_test = df_test["label"].values

# Tokenize and pad with the same tokenizer
X_test_seq = tokenzier.texts_to_sequences(X_test)
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len, padding="post", truncating="post")

# Get predictions
y_test_pred = model_1.predict(X_test_pad)
y_test_pred_classes = y_test_pred.argmax(axis = 1)

# Compute macro F1 
macro_f1 = f1_score(y_test, y_test_pred_classes, average="macro")
print("Macro F1 score on Test Data:", macro_f1)

# Detailed per class evaluation report
print(classification_report(y_test, y_test_pred_classes, target_names=label_encoder.classes_))

In [ ]:
# Get predictions (example for iteration 3 model)
y_test_pred = model_1.predict(X_test_pad).argmax(axis=1)

# Compute confusion matrix
cm = confusion_matrix(y_test, y_test_pred)

# Plot confusion matrix
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_encoder.classes_)
disp.plot(cmap=plt.cm.Blues, xticks_rotation=45)
plt.title("Confusion Matrix (RNN Iteration 3)")
plt.show()

In [ ]:
# Predict on validation set
y_val_pred = model_1.predict(X_val_pad)
y_val_pred_classes = y_val_pred.argmax(axis=1)

# Compute Macro F1
macro_f1_val = f1_score(y_val, y_val_pred_classes, average="macro")
print("Macro F1 Score on Validation Data:", macro_f1_val)

# Full classification report
print(classification_report(y_val, y_val_pred_classes, target_names=label_encoder.classes_))

# Iteration 2: Pre processing + POS tagging

In [ ]:
# Pre-processing texts

def preprocess_text(text):
    # handling missing values
    if pd.isna(text):
        return ""
    text = str(text)
    text = text.lower()

    #Normalize weird unicode (removing things like Ä, non-breaking spaces)
    text = unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode("utf-8", "ignore")

    # Keep letters and punctuation (! ? . , …) but drop numbers/symbols
    text = re.sub(r"[^a-zA-Z!?.,'\s]", " ", text)

    # Removing usernames / mentions
    text = re.sub(r"@\w", " ", text)

    # Removing things like [NAME] or [something]
    text = re.sub(r"\[.*?\]", " ", text)

    # Remove URLs
    text = re.sub(r"http\S+|www\S+", " ", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text


In [ ]:
df = df.dropna(subset=["sentence"])
df = df.drop_duplicates(subset=["sentence"])
df["clean_sentence"] = df["sentence"].apply(preprocess_text)

In [ ]:
import spacy

# Load SpaCy model
nlp = spacy.load("en_core_web_sm")

def get_pos_features(text):
    doc = nlp(text)
    counts = {"ADJ":0, "ADV":0, "NOUN":0, "VERB":0}
    for token in doc:
        if token.pos_ == "ADJ":
            counts["ADJ"] += 1
        elif token.pos_ == "ADV":
            counts["ADV"] += 1
        elif token.pos_ == "NOUN":
            counts["NOUN"] += 1
        elif token.pos_ == "VERB":
            counts["VERB"] += 1
    return list(counts.values())

# Apply to dataset
pos_features = df["clean_sentence"].apply(get_pos_features)
pos_features = np.array(pos_features.tolist())
print(pos_features.shape, pos_features[:5], pos_features.dtype)

In [ ]:
# Tokenize and pad the input

X_seq = tokenzier.texts_to_sequences(df["sentence"].values)
X_pad = pad_sequences(X_seq, maxlen=max_len, padding = "post", truncating="post")

# Split into training and validation sets for both the input and the sentiment scores 
X_train_pad, X_val_pad, pos_train, pos_val, y_train, y_val = train_test_split(
    X_pad, pos_features, df["label"].values, test_size = 0.1,
    random_state=42, stratify=df["label"].values
)

In [ ]:
## Building model

# Text branch
text_in = Input(shape=(max_len), name = "text_input")
x = Embedding(input_dim = max_words, output_dim = embedding_dim, input_length = max_len)(text_in)
x = SimpleRNN(128, return_sequences=False)(x)
x = Dropout(0.5)(x)

# POS branch
pos_in = Input(shape=(pos_train.shape[1],), name="pos_input")
p = Dense(16, activation="relu")(pos_in)

# Combine branches
combined = Concatenate()([x, p])
out = Dense(len(label_encoder.classes_), activation = "softmax")(combined)

# Build model
model_2 = Model(inputs = [text_in, pos_in], outputs = out)

model_2.compile(
    loss = "sparse_categorical_crossentropy",
    optimizer = "adam",
    metrics =["accuracy"]
)

model_2.summary()

In [ ]:
# Flatten out sequence and sentiment features into one array for oversampling
X_train_combined = np.hstack([X_train_pad, pos_train])

ros = RandomOverSampler(random_state=42)
X_resampled, y_resampled = ros.fit_resample(X_train_combined, y_train)

X_train_pad_resampled = X_resampled[:, :X_train_pad.shape[1]]
pos_train_resampled = X_resampled[:, X_train_pad.shape[1]:]

print("Before:", np.bincount(y_train))
print("After:", np.bincount(y_resampled))

In [ ]:
early_stop = EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)

h2 = model_2.fit(
    [X_train_pad_resampled, pos_train_resampled], y_resampled,
    validation_data = ([X_val_pad, pos_val], y_val),
    epochs = 50,
    batch_size = 32,
    callbacks = [early_stop],
    verbose = 1
)

In [ ]:
pos_test= df_test["Sentence"].apply(get_pos_features)
pos_test = np.array(pos_test.tolist())

print(pos_test.shape)
print(pos_test[:5])

In [ ]:
# Get predictions
y_test_pred = model_2.predict([X_test_pad, pos_test])
y_test_pred_classes = y_test_pred.argmax(axis = 1)

# Compute macro F1 
macro_f1 = f1_score(y_test, y_test_pred_classes, average="macro")
print("Macro F1 score on Test Data:", macro_f1)

# Detailed per class evaluation report
print(classification_report(y_test, y_test_pred_classes, target_names=label_encoder.classes_))

In [ ]:
# Get predictions (example for iteration 3 model)
y_test_pred = model_2.predict([X_test_pad, pos_test]).argmax(axis=1)

# Compute confusion matrix
cm = confusion_matrix(y_test, y_test_pred)

# Plot confusion matrix
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_encoder.classes_)
disp.plot(cmap=plt.cm.Blues, xticks_rotation=45)
plt.title("Confusion Matrix (RNN Iteration 3)")
plt.show()

In [ ]:
# Predict on validation set
y_val_pred = model_2.predict([X_val_pad, pos_val])
y_val_pred_classes = y_val_pred.argmax(axis=1)

# Compute Macro F1
macro_f1_val = f1_score(y_val, y_val_pred_classes, average="macro")
print("Macro F1 Score on Validation Data:", macro_f1_val)

# Full classification report
print(classification_report(y_val, y_val_pred_classes, target_names=label_encoder.classes_))

# Iteration 3: POS tagging + sentiment analysis + TF_IDF

In [ ]:
# Sentiment Analysis
nltk.download("vader_lexicon")
sid = SentimentIntensityAnalyzer()

def get_sentiment_scores(text):
    scores = sid.polarity_scores(str(text))
    return [scores["neg"], scores["neu"], scores["pos"], scores["compound"]] # The different scores we want to get

sentiment_features = df["sentence"].apply(get_sentiment_scores)
sentiment_features = np.array(sentiment_features.tolist())

In [ ]:
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2))

X_tfidf = tfidf.fit_transform(df["clean_sentence"]).toarray()
print("TF-IDF shape:", X_tfidf.shape)

In [ ]:
# Load SpaCy model
nlp = spacy.load("en_core_web_sm")

def get_pos_features(text):
    doc = nlp(text)
    counts = {"ADJ":0, "ADV":0, "NOUN":0, "VERB":0}
    for token in doc:
        if token.pos_ == "ADJ":
            counts["ADJ"] += 1
        elif token.pos_ == "ADV":
            counts["ADV"] += 1
        elif token.pos_ == "NOUN":
            counts["NOUN"] += 1
        elif token.pos_ == "VERB":
            counts["VERB"] += 1
    return list(counts.values())

# Apply to dataset
pos_features = df["clean_sentence"].apply(get_pos_features)
pos_features = np.array(pos_features.tolist())
print(pos_features.shape, pos_features[:5], pos_features.dtype)

In [ ]:
# Tokenize and pad the input

X_seq = tokenzier.texts_to_sequences(df["sentence"].values)
X_pad = pad_sequences(X_seq, maxlen=max_len, padding = "post", truncating="post")

# Split into training and validation sets for the input, sentiment scores, and the POS tags
X_train_pad, X_val_pad, sent_train, sent_val, pos_train, pos_val, tfidf_train, tfidf_val, y_train, y_val = train_test_split(
    X_pad, sentiment_features,pos_features, X_tfidf, df["label"].values, test_size = 0.1,
    random_state=42, stratify=df["label"].values
)


In [ ]:
## Building the model

seq_input = Input(shape=(max_len,))
embedding_layer = Embedding(
    input_dim = max_words,
    output_dim = 128,
    input_length = max_len,
    trainable = True
)(seq_input)

# Stacked LSTM layers
x = SimpleRNN(128, return_sequences=True)(embedding_layer)
x = Dropout(0.5)(x)
x = SimpleRNN(64)(x)
x = Dropout(0.3)(x)

# Sentiment input
sent_input = Input(shape=(4,))
dense_sent = Dense(16, activation = "relu")(sent_input)

# POS input
pos_input = Input(shape=(4,))
dense_pos = Dense(16, activation = "relu")(pos_input)


# TF-IDF branch
tfidf_input = Input(shape=(X_tfidf.shape[1],))
dense_tfidf = Dense(128, activation = "relu")(tfidf_input)
dense_tfidf = Dropout(0.3)(dense_tfidf)

# Merge
merged = Concatenate()([x, dense_sent, dense_pos, dense_tfidf])
dense_out = Dense(64, activation = "relu")(merged)
dropout = Dropout(0.3)(dense_out)
final_out = Dense(len(label_encoder.classes_), activation = "softmax")(dropout)

#Build model
model_3 = Model(inputs=[seq_input, pos_input, sent_input, tfidf_input], outputs = final_out)
model_3.compile(loss = "sparse_categorical_crossentropy",
                optimizer = "adam", 
                metrics = ["accuracy"])

model_3.summary()

In [ ]:
# Stack sentiment, POS, TF-IDF for resampling together
extra_train = np.hstack([sent_train, pos_train, tfidf_train])

# Oversample
ros = RandomOverSampler(random_state=42)
extra_resampled, y_train_resampled = ros.fit_resample(extra_train, y_train)

# Get indices of rows that were resampled
indices = ros.sample_indices_

# Apply same resampling to padded sequences
X_train_pad_resampled = X_train_pad[indices]

# Now split extra_resampled back into features
sent_train_resampled = extra_resampled[:, :sent_train.shape[1]]
pos_train_resampled  = extra_resampled[:, sent_train.shape[1]:sent_train.shape[1]+pos_train.shape[1]]
tfidf_train_resampled = extra_resampled[:, sent_train.shape[1]+pos_train.shape[1]:]

In [ ]:
h3 = model_3.fit(
    [X_train_pad_resampled, sent_train_resampled, pos_train_resampled, tfidf_train_resampled], y_resampled,
    validation_data=([X_val_pad, sent_val, pos_val, tfidf_val], y_val),
    epochs=10,        
    batch_size=32,
    callbacks = [early_stop],
    verbose=1
)

In [ ]:
pos_test= df_test["Sentence"].apply(get_pos_features)
pos_test = np.array(pos_test.tolist())

print(pos_test.shape)
print(pos_test[:5])

In [ ]:
tfidf_test  = tfidf.transform(X_test).toarray()
print("TF-IDF shape:", tfidf_test.shape)

In [ ]:
# Generate sentiment features for test sentences
sent_test = df_test["Sentence"].apply(get_sentiment_scores)
sent_test = np.array(sent_test.tolist())

print(sent_test.shape)
print(sent_test[:5])

In [ ]:
# Get predictions
y_test_pred = model_3.predict([X_test_pad, sent_test, pos_test, tfidf_test])
y_test_pred_classes = y_test_pred.argmax(axis = 1)

# Compute macro F1 
macro_f1 = f1_score(y_test, y_test_pred_classes, average="macro")
print("Macro F1 score on Test Data:", macro_f1)

# Detailed per class evaluation report
print(classification_report(y_test, y_test_pred_classes, target_names=label_encoder.classes_))

In [ ]:

# Get predictions (example for iteration 3 model)
y_test_pred = model_3.predict([X_test_pad, sent_test, pos_test, tfidf_test]).argmax(axis=1)

# Compute confusion matrix
cm = confusion_matrix(y_test, y_test_pred)

# Plot confusion matrix
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_encoder.classes_)
disp.plot(cmap=plt.cm.Blues, xticks_rotation=45)
plt.title("Confusion Matrix (RNN Iteration 3)")
plt.show()

In [ ]:
# Predict on validation set
y_val_pred = model_3.predict([X_val_pad, sent_val, pos_val, tfidf_val])
y_val_pred_classes = y_val_pred.argmax(axis=1)

# Compute Macro F1
macro_f1_val = f1_score(y_val, y_val_pred_classes, average="macro")
print("Macro F1 Score on Validation Data:", macro_f1_val)

# Full classification report
print(classification_report(y_val, y_val_pred_classes, target_names=label_encoder.classes_))